# 04 - Classification Models
**Syllabus link:** Logistic Regression, Nearest Neighbors (lazy learning), Naive Bayes, Decision Trees and Rules, Support Vector Machine

**Task:** predict `Class` (1 = fraud). Metrics: precision, recall, F1, ROC-AUC, PR-AUC (accuracy shown only to demonstrate why it misleads).

**Method**
1. Stratified 80/20 train/test split (test set stays untouched and imbalanced, like real data).
2. Scale `Time` and `Amount` with RobustScaler fitted on the training set only.
3. Handle imbalance on the **training set only**: random undersampling of legitimate rows (1 fraud : 10 legit).
4. Train 5 models, tune with GridSearchCV (scoring = average precision), evaluate on the full test set.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.svm import SVC
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, accuracy_score)

sns.set_style("whitegrid")
FIG_DIR = "../reports/figures"
PROC_DIR = "../data/processed"
os.makedirs(FIG_DIR, exist_ok=True)
def savefig(name):
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

SEED = 42
df = pd.read_csv(f"{PROC_DIR}/creditcard_clean.csv")
X = df.drop(columns=["Class", "Hour"])
y = df["Class"]
print(X.shape, y.value_counts().to_dict())

## 1. Stratified split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train:", X_train.shape, "fraud:", y_train.sum(), f"({y_train.mean()*100:.3f}%)")
print("Test :", X_test.shape, "fraud:", y_test.sum(), f"({y_test.mean()*100:.3f}%)")

## 2. Scaling (fit on training data only)

In [ ]:
scaler = RobustScaler().fit(X_train[["Time", "Amount"]])
X_train_s, X_test_s = X_train.copy(), X_test.copy()
X_train_s[["Time", "Amount"]] = scaler.transform(X_train[["Time", "Amount"]])
X_test_s[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

## 3. Imbalance handling: random undersampling (training set only)

In [ ]:
rng = np.random.RandomState(SEED)
fraud_idx = y_train.index[y_train == 1]
legit_idx = y_train.index[y_train == 0]
legit_keep = rng.choice(legit_idx, size=10 * len(fraud_idx), replace=False)
keep = np.concatenate([fraud_idx, legit_keep])
rng.shuffle(keep)

X_res, y_res = X_train_s.loc[keep], y_train.loc[keep]
print("Resampled train:", X_res.shape, y_res.value_counts().to_dict())

Undersampling keeps all fraud cases and a random 10x sample of legitimate ones. Resampling is done **after** the split and **only on training data**, so the test set still reflects real-world imbalance.
Trade-off: legitimate information is discarded; later we also compare class weights on the full training data.

## 4. Evaluation helpers

In [ ]:
def get_scores(model, X_):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_)[:, 1]
    return model.decision_function(X_)

def evaluate(name, model, X_te=None, y_te=None):
    X_te = X_test_s if X_te is None else X_te
    y_te = y_test if y_te is None else y_te
    pred = model.predict(X_te)
    score = get_scores(model, X_te)
    tn, fp, fn, tp = confusion_matrix(y_te, pred).ravel()
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, zero_division=0),
        "Recall": recall_score(y_te, pred),
        "F1": f1_score(y_te, pred),
        "ROC-AUC": roc_auc_score(y_te, score),
        "PR-AUC": average_precision_score(y_te, score),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }, pred, score

all_results, store = [], {}

## 5. Baseline: always predict legitimate

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_res, y_res)
dummy_full = DummyClassifier(strategy="most_frequent").fit(X_train_s, y_train)
r, _, _ = evaluate("Dummy (always legit)", dummy_full)
all_results.append(r)
print(f"Accuracy = {r['Accuracy']:.4f} but Recall = {r['Recall']:.2f}: accuracy is useless here.")

## 6. Five classifiers with default settings

In [ ]:
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=SEED),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "SVM": SVC(kernel="rbf", random_state=SEED),
}

base_rows = []
for name, m in base_models.items():
    m.fit(X_res, y_res)
    r, _, _ = evaluate(f"{name} (default)", m)
    base_rows.append(r)
all_results.extend(base_rows)

base_df = pd.DataFrame(base_rows).set_index("Model").round(4)
base_df

## 7. Hyperparameter tuning (GridSearchCV, scoring = average precision)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

param_grids = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, random_state=SEED),
                            {"C": [0.01, 0.1, 1, 10, 100]}),
    "KNN": (KNeighborsClassifier(),
            {"n_neighbors": [3, 5, 7, 9, 11, 15], "weights": ["uniform", "distance"]}),
    "Naive Bayes": (GaussianNB(),
                    {"var_smoothing": [1e-11, 1e-9, 1e-7, 1e-5]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=SEED),
                      {"max_depth": [3, 4, 5, 6, 8, 10, None], "min_samples_leaf": [1, 5, 10]}),
    "SVM": (SVC(random_state=SEED),
            {"C": [0.1, 1, 10], "kernel": ["linear", "rbf"], "gamma": ["scale"]}),
}

tuned, best_params = {}, {}
for name, (est, grid) in param_grids.items():
    gs = GridSearchCV(est, grid, scoring="average_precision", cv=cv, n_jobs=-1)
    gs.fit(X_res, y_res)
    tuned[name] = gs.best_estimator_
    best_params[name] = gs.best_params_
    print(f"{name:20s} best params: {gs.best_params_} | CV PR-AUC = {gs.best_score_:.4f}")

Note: CV runs on the undersampled training data, so CV scores are optimistic compared with the final test scores. The test set (untouched, imbalanced) gives the realistic estimate.

## 8. Tuned models on the test set

In [ ]:
tuned_rows = []
for name, m in tuned.items():
    r, pred, score = evaluate(f"{name} (tuned)", m)
    tuned_rows.append(r)
    store[name] = {"pred": pred, "score": score}
all_results.extend(tuned_rows)

tuned_df = pd.DataFrame(tuned_rows).set_index("Model").round(4)
tuned_df

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for a, (name, d) in zip(axes, store.items()):
    cm = confusion_matrix(y_test, d["pred"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"])
    a.set_title(name); a.set_xlabel("Predicted"); a.set_ylabel("Actual")
plt.tight_layout(); savefig("04_confusion_matrices"); plt.show()

Reading a confusion matrix: **FN** (fraud predicted legit) = missed fraud, the costly error. **FP** (legit predicted fraud) = false alarm, an inconvenience. Undersampling usually raises recall but lowers precision.

## 9. Alternative imbalance strategy: class weights on the full training set

In [ ]:
lr_cw = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED).fit(X_train_s, y_train)
dt_cw = DecisionTreeClassifier(max_depth=best_params["Decision Tree"]["max_depth"],
                               min_samples_leaf=best_params["Decision Tree"]["min_samples_leaf"],
                               class_weight="balanced", random_state=SEED).fit(X_train_s, y_train)

cw_rows = []
for name, m in [("Logistic Regression (class_weight, full train)", lr_cw),
                ("Decision Tree (class_weight, full train)", dt_cw)]:
    r, _, _ = evaluate(name, m)
    cw_rows.append(r)
all_results.extend(cw_rows)
pd.DataFrame(cw_rows).set_index("Model").round(4)

Compare these rows with the undersampled versions above: class weights keep all information but can lower precision. Explain the trade-off in the report.

## 10. Decision tree rules

In [ ]:
dt_best = tuned["Decision Tree"]
print("Depth:", dt_best.get_depth(), "| Leaves:", dt_best.get_n_leaves())
print(export_text(dt_best, feature_names=list(X_res.columns), max_depth=4))

In [ ]:
plt.figure(figsize=(20, 9))
plot_tree(dt_best, feature_names=list(X_res.columns), class_names=["Legit", "Fraud"],
          filled=True, rounded=True, max_depth=3, fontsize=9)
plt.title("Decision tree (top 3 levels)")
savefig("04_decision_tree"); plt.show()

imp = pd.Series(dt_best.feature_importances_, index=X_res.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7, 4)); imp[::-1].plot(kind="barh"); plt.title("Decision tree: top feature importances")
savefig("04_tree_importance"); plt.show()

## 11. Logistic regression coefficients (interpretation)

In [ ]:
lr_best = tuned["Logistic Regression"]
coef = pd.DataFrame({"feature": X_res.columns, "coef": lr_best.coef_[0]})
coef["odds_ratio"] = np.exp(coef["coef"])
coef.reindex(coef["coef"].abs().sort_values(ascending=False).index).head(10).round(3)

A coefficient above 0 (odds ratio above 1) raises the odds of fraud; below 0 lowers them.

## 12. Save results for notebook 05

In [ ]:
results_df = pd.DataFrame(all_results).set_index("Model")
joblib.dump({"results": results_df, "store": store, "y_test": y_test.values,
             "best_params": best_params}, f"{PROC_DIR}/results.pkl")
results_df.round(4)

Next: `05_model_comparison.ipynb`.